# 10 — Journal revision: every experiment the reviewers will ask for

Run this on **Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).
Everything is checkpointed to Drive, so a disconnect costs nothing: re-run all
cells and each step resumes where it stopped.

| Step | What it adds to the paper | Needs | Approx. time (T4) |
|---|---|---|---|
| 3 | Two extra Kaggle URL corpora, each checked against the core corpora for duplication | Kaggle login | 5 min |
| 4 | **Procedure-aligned benchmark** (both classes collected as full URLs; plus a version where scheme/www./path/query carry no information by construction) | network | 15–40 min |
| 5 | Structural audit + independence check for every corpus | — | 3 min |
| 6 | XGBoost and logistic regression, all feature sets; overlap-controlled transfer; PR-AUC and TPR@FPR; SHAP; Tranco | CPU | 40–70 min |
| 7 | Character n-gram TF-IDF model | CPU | 10–20 min |
| 8 | Character-level CNN on raw URLs | GPU | 30–60 min |
| 9 | Fine-tuned DistilBERT on raw URLs | GPU | 60–120 min |
| 10 | Summary tables and a zip to send back | — | 1 min |



## 1 — Drive, code, packages

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/research/phishing-detection')   # same folder as notebooks 02-09
assert (DRIVE_DIR / 'data' / 'raw' / 'phiusiil' / 'phiusiil.csv').exists(), 'set DRIVE_DIR to your phishing-detection folder'

REPO = Path('/content/phishing-benchmark-audit')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', 'paper-revision',
                    'https://github.com/sandesh20lamichhane/phishing-benchmark-audit', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
%cd /content/phishing-benchmark-audit
!pip -q install tldextract kagglehub transformers   # xgboost and scikit-learn: Colab's preinstalled versions
import xgboost, sklearn; print('xgboost', xgboost.__version__, '| scikit-learn', sklearn.__version__)

Mounted at /content/drive
/content/phishing-benchmark-audit
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 5.2 MB/s eta 0:00:00
xgboost 3.4.1 | scikit-learn 1.6.1


In [2]:
RAW = DRIVE_DIR / 'data' / 'raw'
os.environ.update({
    'RAW_DIR':   str(RAW),
    'EXTRA_DIR': str(RAW / 'extra'),
    'TRANCO':    str(DRIVE_DIR / 'data' / 'external' / 'tranco' / 'tranco_top1m.csv'),
    'OUT_DIR':   str(DRIVE_DIR / 'results_q1'),
    'CACHE_DIR': str(DRIVE_DIR / 'checkpoints' / 'q1_cache'),
})
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (steps 8-9 will be slow)')

GPU: Tesla T4


## 2 — Verify the raw files are the published ones

In [3]:
!cd "$RAW_DIR" && sha256sum -c /content/phishing-benchmark-audit/checksums.txt 2>/dev/null | grep -E 'phiusiil|hannousse|kaggle_malicious|mendeley_phishurl'

phiusiil/phiusiil.csv: OK
hannousse/hannousse.csv: OK
kaggle_malicious/kaggle_malicious_urls.csv: OK
mendeley_phishurl/mendeley_phishurl_urls.csv: OK


## 3 — Extra Kaggle corpora (optional)

Uses the same Kaggle login as notebook 06. Check the printed label values: rows
with values outside the known good/bad, benign/malicious, legitimate/phishing
sets are dropped, never guessed. Each corpus is then compared with the full
core files: one whose URL containment with an existing corpus exceeds 0.5 is
the same data under another name, so it is moved to `data/raw/duplicates/` and
stays out of the experiments (the table goes into the results zip).

In [4]:
!python scripts/fetch_extra_datasets.py

[kaggle_phishing_site_urls] already present
[kaggle_malicious_benign_urls] already present
[aligned_matched] max containment 0.006 (phiusiil): built here, kept
[aligned_natural] max containment 0.006 (phiusiil): built here, kept


## 4 — Procedure-aligned benchmark (the Q1 contribution)

Phishing URLs from Phishing.Database; legitimate URLs from Common Crawl captures
of domains sampled evenly from three Tranco rank bands. Same per-domain cap for
both classes. Writes `aligned_natural.csv` and `aligned_matched.csv` (the
latter equalises HTTPS, www., empty path and query across classes, so those
properties carry zero information).

Common Crawl lookups read the crawl's own index files from data.commoncrawl.org
(one index table downloaded once, then a few hundred kB per domain), not the
rate-limited index server that stalled the first attempt. Every domain is cached, so re-running the cell resumes. The
progress line counts domains with captures, without, and failed; if more
than three quarters of the first 200 fail, the script stops and says so.

In [5]:
!python scripts/build_aligned_benchmark.py --domains 3000 --per-domain 5 --workers 4

[phish] 789,052 http(s) links -> 323,399 after dedup and per-domain cap 5
[legit] 3,000 Tranco domains, crawl CC-MAIN-2026-39, 886,074 index blocks
  200/3,000 domains: 130 with captures, 70 without, 0 failed [0.6 min]
  400/3,000 domains: 266 with captures, 134 without, 0 failed [1.0 min]
  600/3,000 domains: 401 with captures, 199 without, 0 failed [1.6 min]
  800/3,000 domains: 531 with captures, 269 without, 0 failed [2.1 min]
  1,000/3,000 domains: 660 with captures, 340 without, 0 failed [2.5 min]
  1,200/3,000 domains: 772 with captures, 428 without, 0 failed [3.0 min]
  1,400/3,000 domains: 884 with captures, 516 without, 0 failed [3.4 min]
  1,600/3,000 domains: 1,006 with captures, 594 without, 0 failed [3.8 min]
  1,800/3,000 domains: 1,124 with captures, 676 without, 0 failed [4.3 min]
  2,000/3,000 domains: 1,248 with captures, 752 without, 0 failed [4.7 min]
  2,200/3,000 domains: 1,355 with captures, 845 without, 0 failed [5.1 min]
  2,400/3,000 domains: 1,468 with captu

## 5 — Structural audit and independence of every corpus

Read the containment table: any extra corpus with containment > 0.5 against
another corpus is the same data. Delete its CSV from `data/raw/extra/` and
re-run this cell before continuing.

In [6]:
!python scripts/revision_experiments.py audit

== audit
          dataset class     n  scheme_present_pct  https_pct  empty_path_pct  query_pct  www_pct
         phiusiil legit 34397              100.00     100.00          100.00       0.00   100.00
         phiusiil phish 25603              100.00      48.55           40.39       5.96    41.34
mendeley_phishurl legit 41067              100.00      99.99            0.74      13.56    99.83
mendeley_phishurl phish 18933              100.00      34.80            5.02      16.20     9.55
 kaggle_malicious legit 49169                8.30       0.48            0.07      17.12     3.20
 kaggle_malicious phish 10831               26.01       7.68           23.40       8.41    38.65
        hannousse legit  5715              100.00      44.50            6.16       3.34    66.95
        hannousse phish  5715              100.00      33.32            5.02      23.67    20.70
         ebbu2017 legit 29640              100.00      36.75            0.00       4.08    58.55
         ebbu2017 phi

## 6 — Feature-based models, overlap control, operating-point metrics, SHAP, Tranco

In [7]:
!python scripts/revision_experiments.py xgb www lr shap tranco mendeley

== xgb
== www
== lr
== shap
  [shap] phiusiil
  [shap] mendeley_phishurl
  [shap] kaggle_malicious
  [shap] hannousse
  [shap] ebbu2017
  [shap] aligned_matched
  [shap] aligned_natural
== tranco
          dataset class     n  bare_root_pct  in_tranco_top10k_pct  in_tranco_top100k_pct  in_tranco_top1000k_pct
         phiusiil legit 34397         100.00                  2.56                  10.14                   32.90
         phiusiil phish 25603          72.96                 24.59                  40.10                   45.00
mendeley_phishurl legit 41067          14.93                 35.75                  52.30                   66.79
mendeley_phishurl phish 18933          21.70                 18.41                  28.28                   34.27
 kaggle_malicious legit 49169          12.43                 33.93                  49.47                   62.28
 kaggle_malicious phish 10831          25.74                 22.94                  33.63                   42.18
      

## 7 — Character n-gram TF-IDF + logistic regression (raw string)

In [8]:
!python scripts/revision_experiments.py tfidf

== tfidf
  [tfidf_lr/raw_string] kaggle_malicious seed 42 [0.4 min]
  [tfidf_lr/raw_string] hannousse seed 42 [0.6 min]
  [tfidf_lr/raw_string] ebbu2017 seed 42 [1.0 min]
  [tfidf_lr/raw_string] aligned_matched seed 42 [1.6 min]
  [tfidf_lr/raw_string] aligned_natural seed 42 [2.7 min]


## 8 — Character-level CNN (GPU)

In [9]:
!python scripts/char_cnn.py

device: cuda
  [char_cnn] phiusiil seed 42 [0.5 min]
  [char_cnn] phiusiil seed 43 [0.9 min]
  [char_cnn] phiusiil seed 44 [1.3 min]
  [char_cnn] mendeley_phishurl seed 42 [1.7 min]
  [char_cnn] mendeley_phishurl seed 43 [2.1 min]
  [char_cnn] mendeley_phishurl seed 44 [2.5 min]
  [char_cnn] kaggle_malicious seed 42 [2.9 min]
  [char_cnn] kaggle_malicious seed 43 [3.3 min]
  [char_cnn] kaggle_malicious seed 44 [3.7 min]
  [char_cnn] hannousse seed 42 [3.8 min]
  [char_cnn] hannousse seed 43 [3.9 min]
  [char_cnn] hannousse seed 44 [4.0 min]
  [char_cnn] ebbu2017 seed 42 [4.4 min]
  [char_cnn] ebbu2017 seed 43 [4.8 min]
  [char_cnn] ebbu2017 seed 44 [5.2 min]
  [char_cnn] aligned_matched seed 42 [5.4 min]
  [char_cnn] aligned_matched seed 43 [5.7 min]
  [char_cnn] aligned_matched seed 44 [5.9 min]
  [char_cnn] aligned_natural seed 42 [6.2 min]
  [char_cnn] aligned_natural seed 43 [6.5 min]
  [char_cnn] aligned_natural seed 44 [6.8 min]


## 9 — Fine-tuned DistilBERT (GPU)

One seed by default. Add seeds as arguments (e.g. `42 43 44`) if time allows.

In [ ]:
!python scripts/transformer_url.py 42 43 44 45 46

model distilbert-base-uncased on cuda, seeds (42,)
config.json: 100% 483/483 [00:00<00:00, 1.90MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 277kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 6.11MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 12.9MB/s]

model.safetensors: downloading bytes:  80% 215M/268M [00:01<00:00, 325MB/s, 16.2MB/s  ]
model.safetensors: reconstructing file:   1% 1.58M/268M [00:01<03:26, 1.29MB/s]
model.safetensors: downloading bytes: 100% 250M/250M [00:01<00:00, 164MB/s, 23.6MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:01<00:00, 176MB/s, 25.7MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 11256.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTE

## 10 — Summary and the zip to send back

In [ ]:
!python scripts/summarise_revision.py
import shutil, json
for f in ['aligned_manifest.json']:
    p = RAW / f
    if p.exists(): shutil.copy2(p, Path(os.environ['OUT_DIR']) / f)
zip_path = shutil.make_archive(str(DRIVE_DIR / 'q1_results'), 'zip', os.environ['OUT_DIR'])
print('Upload this file to the project thread:', zip_path)